In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

TABLES_DIR = Path("../results/tables")
FIGURES_DIR = Path("../results/figures")

FIGURES_DIR.mkdir(
    parents = True,
    exist_ok = True
)

comparison_results = pd.DataFrame(
    {
        "model": [
            "One-Class SVM",
            "Isolation Forest",
            "Autoencoder"
        ],
        
        "accuracy": [
            0.988095,
            0.948209,
            0.983959
        ],
        
        "precision": [
            0.997318,
            0.997457,
            0.997649
        ],
        
        "recall": [
            0.990218,
            0.948276,
            0.985553
        ],
        
        "f1": [
            0.993755,
            0.972245,
            0.991564
        ],
        
        "balanced_accuracy": [
            0.965771,
            0.947505,
            0.967184
        ],
        
        "fpr": [
            0.058677,
            0.053267,
            0.051186
        ],
        
        "roc_auc": [
            0.988331,
            0.959372,
            0.992868
        ],
        
        "pr_auc": [
            0.999391,
            0.994644,
            0.999650
        ]
    }
)

comparison_results

In [ ]:
comparison_ranked = (
    comparison_results.sort_values(
        by = [
            "balanced_accuracy",
            "fpr",
            "recall"
        ],
        ascending = [
            False,
            True,
            False
        ]
    )
    .reset_index(drop = True)
)

comparison_ranked

In [ ]:
confusion_comparison = pd.DataFrame(
    {
        "model": [
            "One-Class SVM",
            "Isolation Forest",
            "Autoencoder"
        ],
        
        "tn": [
            2262,
            2275,
            2280
        ],
        
        "fp": [
            141,
            128,
            123
        ],
        
        "fn": [
            518,
            2739,
            765
        ],
        
        "tp": [
            52436,
            50215,
            52189
        ]
    }
)
confusion_comparison

In [ ]:
confusion_comparison["normal_total"] = (
    confusion_comparison["tn"] 
    + confusion_comparison["fp"]
)

confusion_comparison["anomaly_total"] = (
    confusion_comparison["fn"]
    + confusion_comparison["tp"]
)

confusion_comparison["total"] = (
    confusion_comparison["normal_total"]
    + confusion_comparison["anomaly_total"]
)

confusion_comparison

In [ ]:
metrics_to_plot = [
    "accuracy",
    "recall",
    "f1",
    "balanced_accuracy",
    "roc_auc",
    "pr_auc"
]

plot_data = (
    comparison_results
    .set_index("model")[metrics_to_plot]
    .T
)

plot_data.plot(
    kind = "bar",
    figsize=(12, 6)
)

plt.ylabel("Score")
plt.xlabel("Metric")
plt.title("Final Model Performance Comparison")

plt.ylim(
    0.90,
    1.01
)

plt.xticks(
    rotation = 45,
    ha = "right"
)

plt.grid(
    axis = "y"
)

plt.legend(
    title = "Model"
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "final_model_performance.png",
    dpi = 300,
    bbox_inches = "tight"
)

plt.show()

In [ ]:
plt.figure(
    figsize=(8,5)
)

plt.bar(
    comparison_results["model"],
    comparison_results["fpr"]
)

plt.ylabel("False Positive Rate")
plt.title("False Positive Rate Comparison")

plt.grid(
    axis = "y"
)

plt.tight_layout()

In [ ]:
attack_files = sorted(
    TABLES_DIR.glob(
        "*attack_detection*.csv"
    )
)

for file in attack_files:
    print(file.name)

In [ ]:
svm_attack = pd.read_csv(
    TABLES_DIR
    / "one_class_svm_attack_detection.csv"
)

if_attack = pd.read_csv(
    TABLES_DIR
    / "isolation_forest_attack_detection.csv"
)

ae_attack = pd.read_csv(
    TABLES_DIR
    / "autoencoder_attack_detection.csv"
)


print(
    "One-Class SVM:",
    svm_attack.shape
)

print(
    "Isolation Forest:",
    if_attack.shape
)

print(
  "Autoencoder:",
  ae_attack.shape  
)

In [ ]:
svm_attack = (
    svm_attack[
       [
          "Attack_type",
          "detection_rate"
        ] 
    ]
    .rename(
        columns={
            "detection_rate":
            "One-Class SVM"    
        }
    )
)

if_attack = (
    if_attack[
        [
            "Attack_type",
            "detection_rate"
        ] 
    ]
    .rename(
        columns={
            "detection_rate":
            "Isolation Forest"   
        }
    )
)

ae_attack = (
    ae_attack [
        [
            "Attack_type",
            "detection_rate"    
        ]
    ]
    .rename(
        columns={
            "detection_rate":
            "Autoencoder"    
        }
    )
)

In [ ]:
attack_comparison = (
    svm_attack
    .merge(
        if_attack,
        on = "Attack_type"
    )
    .merge(
        ae_attack,
        on = "Attack_type"
    )
)

attack_comparison

In [ ]:
x = np.arange(
    len(attack_comparison)
)

width = 0.25

plt.figure(
    figsize=(14,6)
)

plt.bar(
    x - width,
    attack_comparison[
        "One-Class SVM"
    ],
    width,
    label = "One-Class SVM"
)

plt.bar(
    x,
    attack_comparison[
        "Isolation Forest"
    ],
    width,
    label = "Isolation Forest"
)

plt.bar(
    x + width,
    attack_comparison[
        "Autoencoder"
    ],
    width,
    label = "Autoencoder"
)

plt.xticks(
    x,
    attack_comparison[
        "Attack_type"
    ],
    rotation = 45,
    ha = "right"
)

plt.ylabel(
    "Detection Rate"
)

plt.title(
    "Detection Rate by Attack Type"
)

plt.ylim(
    0,
    1.05
)

plt.legend()

plt.grid(
    axis = "y"
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR
    / "attack_detection_comparison.png",
    dpi = 300,
    bbox_inches = "tight"
)

plt.show()

In [ ]:
metric_directions = {
    "accuracy": "max", 
    "precision": "max",
    "recall":"max",
    "f1": "max",
    "balanced_accuracy": "max",
    "fpr": "min",
    "roc_auc": "max",
    "pr_auc": "max"
}

best_rows = []

for metric, direction in (
    metric_directions.items()
):
    if direction == "max":
        best_index = (
            comparison_results[
                metric
            ].idxmax()
        )
    
    else:
        best_index = (
            comparison_results[
                metric
            ].idxmin()
        )   
    
    best_rows.append(
        {
            "metric": metric,
            "best_model":
            comparison_results.loc[
                best_index,
                "model"
            ],
            "best_value":
            comparison_results.loc[
                best_index,
                metric
            ]
        }
    )

best_models = pd.DataFrame(
    best_rows
)

best_models         

In [ ]:
best_overall = (
    comparison_ranked.iloc[0]
)

print(
    "Best overall model:",
    best_overall["model"]
)

print(
    "Balanced Accuracy:",
    best_overall[
        "balanced_accuracy"
    ]
)

print(
    "Recall:",
    best_overall["recall"]
)

print(
    "False Positive Rate:",
    best_overall["fpr"]
)

In [ ]:
comparison_results.to_csv(
    TABLES_DIR
    / "final_model_comparison.csv",
    index = False
)

comparison_ranked.to_csv(
    TABLES_DIR
    / "final_model_ranking.csv",
    index = False
)

confusion_comparison.to_csv(
    TABLES_DIR
    / "final_confusion_comparison.csv",
    index = False
)
attack_comparison.to_csv(
    TABLES_DIR
    / "final_attack_comparison.csv",
    index = False
)


best_models.to_csv(
    TABLES_DIR
    / "final_best_models.csv",
    index = False
)